# Reasoning traces with the chess dataset — Qwen3-4B-Instruct

In [38]:
from datasets import load_dataset

ds = load_dataset("bonna46/Chess-FEN-and-NL-Format-30K-Dataset")

In [39]:
ds

DatasetDict({
    train: Dataset({
        features: ['FEN', 'Next move', 'FEN + Next move', 'NL Format of FEN'],
        num_rows: 30862
    })
})

In [40]:
from dotenv import load_dotenv
import os
from pathlib import Path

load_dotenv()

hf_root = Path(os.environ['HF_ROOT'])
hf_root.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(hf_root)
os.environ["TRANSFORMERS_CACHE"] = str(hf_root / "transformers")
os.environ["HF_DATASETS_CACHE"] = str(hf_root / "datasets")

In [41]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    trust_remote_code=True,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

print(f"Loaded {MODEL_ID} on {device} with dtype={dtype}")

Loading weights: 100%|██████████| 398/398 [00:00<00:00, 5402.58it/s]


Loaded Qwen/Qwen3-4B-Instruct-2507 on cuda with dtype=torch.bfloat16


In [42]:
def predict_next_move(fen, max_new_tokens=512):
    prompt = (
        "You are a chess expert. Given the position in FEN notation below, "
        "determine the best next move.\n\n"
        f"FEN: {fen}\n\n"
        "Output only the next move in standard algebraic notation (e.g. e4, Nf3, O-O). "
        "No explanation."
    )
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(text, return_tensors="pt").to(device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    ).strip()
    return response

In [43]:
import pandas as pd

ds_train = ds["train"]
results = []

for i in range(10):
    row = ds_train[i]
    fen = row["FEN"]
    actual_move = row["Next move"]

    predicted_move = predict_next_move(fen)

    results.append({
        "row": i,
        "fen": fen,
        "actual_move": actual_move,
        "predicted_move": predicted_move,
    })
    print(f"Row {i}: actual={actual_move!r}  predicted={predicted_move!r}")

df_chess = pd.DataFrame(results)
df_chess[["row", "fen", "actual_move", "predicted_move"]]

Row 0: actual='a8a4'  predicted='Bf6'
Row 1: actual='b7a6'  predicted='Nf6'
Row 2: actual='b4c5'  predicted='e4'
Row 3: actual='c3b5'  predicted='Nf3'
Row 4: actual='a6a5'  predicted='Qd5'
Row 5: actual='a3b5'  predicted='Qe2'
Row 6: actual='c1f4'  predicted='Qe2'
Row 7: actual='f1g2'  predicted='Nf3'
Row 8: actual='c8b7'  predicted='Qf6'
Row 9: actual='b6e6'  predicted='e4'


,row,fen,actual_move,predicted_move
0,0,rnb2k1q/3ppp2/7b/1N3Bpr/QP3Pn1/P7/2PPN1R1/1RBK...,a8a4,Bf6
1,1,1rbq1b1r/ppp1k3/Q1npp1p1/5p1p/1PPP4/P3P1P1/R1B...,b7a6,Nf6
2,2,r2qkbnr/p1pbpppp/3p4/1pn5/1P4P1/2NP4/P1P1PP1P/...,b4c5,e4
3,3,rnb5/3pbp2/3k1n1r/pp4pp/2P2Pq1/K1NpB2N/PP2P1BP...,c3b5,Nf3
4,4,r3kbnr/7p/qpnp1pp1/R1p1p3/4P1QP/5NPR/1PPP1P2/1...,a6a5,Qd5
5,5,r2k1b2/p1qppp1r/npbp3p/P1n1B3/1RP3p1/NP1P3P/4P...,a3b5,Qe2
6,6,2rk1bnr/N3pp2/1p2b3/1pp3Pp/5q2/P1NP4/R1P1P1P1/...,c1f4,Qe2
7,7,r1bqkr2/ppppnp2/2n1p1p1/P3N3/1bP1P3/3P1P2/1PQ1...,f1g2,Nf3
8,8,rnbqkbr1/2p1pp1p/1p1p1np1/pP1P4/P6P/1QP5/4PPPR...,c8b7,Qf6
9,9,2b1kbnr/rpp2p2/pQ2q3/3pp1pp/1n1P4/2P2N2/PP2PPP...,b6e6,e4


In [44]:
SEP = "-" * 60

for _, row in df_chess.iterrows():
    print(f"=== Row {int(row['row'])} ===")
    print(f"FEN:       {row['fen']}")
    print(f"Actual:    {row['actual_move']}")
    print(f"Predicted: {row['predicted_move']}")
    print(SEP)

=== Row 0 ===
FEN:       rnb2k1q/3ppp2/7b/1N3Bpr/QP3Pn1/P7/2PPN1R1/1RBK4 b - - 2 24
Actual:    a8a4
Predicted: Bf6
------------------------------------------------------------
=== Row 1 ===
FEN:       1rbq1b1r/ppp1k3/Q1npp1p1/5p1p/1PPP4/P3P1P1/R1B1KP1P/1NB3NR b - - 1 19
Actual:    b7a6
Predicted: Nf6
------------------------------------------------------------
=== Row 2 ===
FEN:       r2qkbnr/p1pbpppp/3p4/1pn5/1P4P1/2NP4/P1P1PP1P/R1BQKBNR w KQkq - 1 6
Actual:    b4c5
Predicted: e4
------------------------------------------------------------
=== Row 3 ===
FEN:       rnb5/3pbp2/3k1n1r/pp4pp/2P2Pq1/K1NpB2N/PP2P1BP/R5R1 w - - 0 21
Actual:    c3b5
Predicted: Nf3
------------------------------------------------------------
=== Row 4 ===
FEN:       r3kbnr/7p/qpnp1pp1/R1p1p3/4P1QP/5NPR/1PPP1P2/1NB1KB2 b kq - 2 12
Actual:    a6a5
Predicted: Qd5
------------------------------------------------------------
=== Row 5 ===
FEN:       r2k1b2/p1qppp1r/npbp3p/P1n1B3/1RP3p1/NP1P3P/4P1B1/4K1N1 w - - 0 24

In [45]:
first_fen = ds["train"][0]["FEN"]
first_actual = ds["train"][0]["Next move"]

# No max_new_tokens cap — let the model stop on its own EOS up to its full context window
prompt = (
    "You are a chess expert. Given the position in FEN notation below, "
    "determine the best next move.\n\n"
    f"FEN: {first_fen}\n\n"
    "Output only the next move in standard algebraic notation (e.g. e4, Nf3, O-O). "
    "No explanation."
)
messages = [{"role": "user", "content": prompt}]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)
inputs = tokenizer(text, return_tensors="pt").to(device)

remaining_ctx = model.config.max_position_embeddings - inputs["input_ids"].shape[1]

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=remaining_ctx,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

response_full = tokenizer.decode(
    output[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True,
).strip()

print(f"FEN:       {first_fen}")
print(f"Actual:    {first_actual}")
print(f"Predicted: {response_full}")

FEN:       rnb2k1q/3ppp2/7b/1N3Bpr/QP3Pn1/P7/2PPN1R1/1RBK4 b - - 2 24
Actual:    a8a4
Predicted: Bf6


In [46]:
from tqdm import tqdm

In [47]:
def summarize_partial_response(partial_response):
    prompt = (
        "Summarize the following partial chess reasoning in 2-3 sentences, "
        "preserving the key candidate moves and insights reached so far:\n\n"
        f"{partial_response}"
    )
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )
    inputs = tokenizer(text, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=400,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()


first_fen = ds["train"][0]["FEN"]
first_actual = ds["train"][0]["Next move"]

MAX_ITER = 5
MAX_NEW_TOKENS = 1024

base_prompt = (
    "You are a chess expert. Given the position in FEN notation below, "
    "determine the best next move.\n\n"
    f"FEN: {first_fen}\n\n"
    "Output only the next move in standard algebraic notation (e.g. e4, Nf3, O-O). "
    "No explanation."
)

current_prompt = base_prompt
iteration_log = []
final_response = None

for iteration in tqdm(range(MAX_ITER)):
    print(f"\n=== Iteration {iteration + 1} ===")

    messages = [{"role": "user", "content": current_prompt}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )
    inputs = tokenizer(text, return_tensors="pt").to(device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    ).strip()

    # Instruct model gives a direct response; treat it as final unless it hit the token limit.
    n_new = output[0].shape[0] - inputs["input_ids"].shape[1]
    if n_new < MAX_NEW_TOKENS:
        final_response = generated
        iteration_log.append({"iteration": iteration + 1, "status": "done", "response_chars": len(generated)})
        print(f"Response concluded. Predicted move: {generated!r}")
        break

    print(f"Hit token limit ({len(generated)} chars). Summarizing...")
    summary = summarize_partial_response(generated)
    print(f"Summary: {summary}")
    iteration_log.append({"iteration": iteration + 1, "status": "continuing", "response_chars": len(generated), "summary": summary})

    current_prompt = (
        base_prompt + "\n\n"
        f"Previous attempt summary (iteration {iteration + 1}): {summary}\n"
        "Give your final answer now."
    )
else:
    print("\nMax iterations reached without a final answer.")
    final_response = None

print(f"\nFEN:       {first_fen}")
print(f"Actual:    {first_actual}")
print(f"Predicted: {final_response}")
print(f"\nIterations: {len(iteration_log)}")
for entry in iteration_log:
    print(entry)

  0%|          | 0/5 [00:00<?, ?it/s]


=== Iteration 1 ===


  0%|          | 0/5 [00:00<?, ?it/s]

Response concluded. Predicted move: 'Bf6'

FEN:       rnb2k1q/3ppp2/7b/1N3Bpr/QP3Pn1/P7/2PPN1R1/1RBK4 b - - 2 24
Actual:    a8a4
Predicted: Bf6

Iterations: 1
{'iteration': 1, 'status': 'done', 'response_chars': 3}


The summaries are so similar... was no reasoning progress made?

In [48]:
import torch.nn.functional as F
import matplotlib.pyplot as plt

def rolling_slope(y, window=20):
    slopes = []
    x = torch.arange(window, dtype=torch.float32, device=y.device)
    x = x - x.mean()
    denom = (x ** 2).sum()
    for i in range(len(y) - window + 1):
        segment = y[i:i+window]
        segment = segment - segment.mean()
        slope = (x * segment).sum() / denom
        slopes.append(slope)
    return torch.stack(slopes)


first_fen = ds["train"][0]["FEN"]

prompt = (
    "You are a chess expert. Given the position in FEN notation below, "
    "determine the best next move.\n\n"
    f"FEN: {first_fen}\n\n"
    "Output only the next move in standard algebraic notation (e.g. e4, Nf3, O-O). "
    "No explanation."
)
messages = [{"role": "user", "content": prompt}]
text = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True,
)
inputs = tokenizer(text, return_tensors="pt").to(device)

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=1024,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
        return_dict_in_generate=True,
        output_scores=True,
    )

generated_ids = output.sequences[0][inputs["input_ids"].shape[1]:]
scores = torch.stack(output.scores).squeeze(1)
log_probs = F.log_softmax(scores, dim=-1)
token_log_probs = log_probs.gather(1, generated_ids.unsqueeze(1)).squeeze(1).float()
decoded_tokens = [tokenizer.decode([tid.item()]) for tid in generated_ids]
slopes = rolling_slope(token_log_probs, window=20)
slope_x = torch.arange(len(slopes)).cpu()

print(f"Generated {len(generated_ids)} tokens")
print(f"Mean log prob:  {token_log_probs.mean().item():.4f}")

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
axes[0].plot(token_log_probs.cpu().numpy(), linewidth=0.7, alpha=0.85, color="tab:blue")
axes[0].set_ylabel("Log prob")
axes[0].set_title("Per-token log probability")
axes[1].plot(slope_x.numpy(), slopes.cpu().numpy(), linewidth=0.7, color="tab:orange")
axes[1].axhline(0, color="k", linestyle="--", linewidth=0.6)
axes[1].set_xlabel("Token index")
axes[1].set_ylabel("Slope")
axes[1].set_title("Rolling log prob slope (window=20)")
plt.tight_layout()
plt.show()

RuntimeError: stack expects a non-empty TensorList

In [ ]:
from IPython.display import HTML, display
import matplotlib.colors as mcolors
import numpy as np

WINDOW = 20

# Center-align each slope value to the middle token of its window
half = WINDOW // 2
T = len(decoded_tokens)
slope_aligned = np.full(T, np.nan)
for i, s in enumerate(slopes.cpu().numpy()):
    center = i + half
    if center < T:
        slope_aligned[center] = s

# Symmetric diverging colormap: blue = negative slope, red = positive slope
valid = slope_aligned[~np.isnan(slope_aligned)]
vmax = float(np.abs(valid).max()) if len(valid) else 1.0
norm = mcolors.TwoSlopeNorm(vmin=-vmax, vcenter=0.0, vmax=vmax)
cmap = plt.get_cmap("RdBu_r")

def slope_to_hex(s_val):
    if np.isnan(s_val):
        return "#d8d8d8"
    return mcolors.to_hex(cmap(norm(float(s_val))))

# Build one colored <span> per token
parts = []
for tok, s_val in zip(decoded_tokens, slope_aligned):
    display_tok = (
        tok.replace("&", "&amp;")
           .replace("<", "&lt;")
           .replace(">", "&gt;")
           .replace("\n", "↵<br>")
    )
    tooltip = f"{s_val:.5f}" if not np.isnan(s_val) else "n/a"
    bg = slope_to_hex(s_val)
    # darken text for very light backgrounds
    fg = "#000"
    parts.append(
        f'<span title="slope={tooltip}" style="background:{bg};color:{fg};'
        f'padding:1px 3px;border-radius:2px;font-family:monospace;font-size:13px;">'
        f'{display_tok}</span>'
    )

html_body = (
    '<p style="font-family:sans-serif;font-size:12px;margin-bottom:4px;">'
    '<b>Token log-prob slope</b> &mdash; '
    '<span style="background:#2166ac;color:white;padding:1px 5px;border-radius:2px;">blue</span> '
    'slope &lt; 0 (confidence falling) &nbsp;|&nbsp; '
    '<span style="background:#d6604d;color:white;padding:1px 5px;border-radius:2px;">red</span> '
    'slope &gt; 0 (confidence rising) &nbsp;|&nbsp; '
    '<span style="background:#d8d8d8;padding:1px 5px;border-radius:2px;">gray</span> '
    'no window yet</p>'
    '<div style="line-height:2.6;word-wrap:break-word;white-space:pre-wrap;border:1px solid #ccc;'
    'padding:10px;border-radius:4px;max-height:600px;overflow-y:auto;">'
    + "".join(parts)
    + "</div>"
)

display(HTML(html_body))

In [ ]:
slopes = rolling_slope(token_log_probs, window=10)
slope_x = torch.arange(len(slopes)).cpu()

print(f"Generated {len(generated_ids)} tokens")
print(f"Mean log prob:  {token_log_probs.mean().item():.4f}")

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
axes[0].plot(token_log_probs.cpu().numpy(), linewidth=0.7, alpha=0.85, color="tab:blue")
axes[0].set_ylabel("Log prob")
axes[0].set_title("Per-token log probability")
axes[1].plot(slope_x.numpy(), slopes.cpu().numpy(), linewidth=0.7, color="tab:orange")
axes[1].axhline(0, color="k", linestyle="--", linewidth=0.6)
axes[1].set_xlabel("Token index")
axes[1].set_ylabel("Slope")
axes[1].set_title("Rolling log prob slope (window=10)")
plt.tight_layout()
plt.show()

In [ ]:
from IPython.display import HTML, display
import matplotlib.colors as mcolors
import numpy as np

WINDOW = 10

# Center-align each slope value to the middle token of its window
half = WINDOW // 2
T = len(decoded_tokens)
slope_aligned = np.full(T, np.nan)
for i, s in enumerate(slopes.cpu().numpy()):
    center = i + half
    if center < T:
        slope_aligned[center] = s

# Symmetric diverging colormap: blue = negative slope, red = positive slope
valid = slope_aligned[~np.isnan(slope_aligned)]
vmax = float(np.abs(valid).max()) if len(valid) else 1.0
norm = mcolors.TwoSlopeNorm(vmin=-vmax, vcenter=0.0, vmax=vmax)
cmap = plt.get_cmap("RdBu_r")

def slope_to_hex(s_val):
    if np.isnan(s_val):
        return "#d8d8d8"
    return mcolors.to_hex(cmap(norm(float(s_val))))

# Build one colored <span> per token
parts = []
for tok, s_val in zip(decoded_tokens, slope_aligned):
    display_tok = (
        tok.replace("&", "&amp;")
           .replace("<", "&lt;")
           .replace(">", "&gt;")
           .replace("\n", "↵<br>")
    )
    tooltip = f"{s_val:.5f}" if not np.isnan(s_val) else "n/a"
    bg = slope_to_hex(s_val)
    # darken text for very light backgrounds
    fg = "#000"
    parts.append(
        f'<span title="slope={tooltip}" style="background:{bg};color:{fg};'
        f'padding:1px 3px;border-radius:2px;font-family:monospace;font-size:13px;">'
        f'{display_tok}</span>'
    )

html_body = (
    '<p style="font-family:sans-serif;font-size:12px;margin-bottom:4px;">'
    '<b>Token log-prob slope</b> &mdash; '
    '<span style="background:#2166ac;color:white;padding:1px 5px;border-radius:2px;">blue</span> '
    'slope &lt; 0 (confidence falling) &nbsp;|&nbsp; '
    '<span style="background:#d6604d;color:white;padding:1px 5px;border-radius:2px;">red</span> '
    'slope &gt; 0 (confidence rising) &nbsp;|&nbsp; '
    '<span style="background:#d8d8d8;padding:1px 5px;border-radius:2px;">gray</span> '
    'no window yet</p>'
    '<div style="line-height:2.6;word-wrap:break-word;white-space:pre-wrap;border:1px solid #ccc;'
    'padding:10px;border-radius:4px;max-height:600px;overflow-y:auto;">'
    + "".join(parts)
    + "</div>"
)

display(HTML(html_body))

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

def representation_drift(hidden_states):
    h_prev = hidden_states[:-1]
    h_next = hidden_states[1:]
    l2_drift = torch.norm(h_next - h_prev, dim=-1)
    cosine_drift = 1 - F.cosine_similarity(h_next, h_prev, dim=-1)
    return {"l2": l2_drift, "cosine": cosine_drift}


first_fen = ds["train"][0]["FEN"]

prompt = (
    "You are a chess expert. Given the position in FEN notation below, "
    "determine the best next move.\n\n"
    f"FEN: {first_fen}\n\n"
    "Output only the next move in standard algebraic notation (e.g. e4, Nf3, O-O). "
    "No explanation."
)
messages = [{"role": "user", "content": prompt}]
text = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True,
)
inputs = tokenizer(text, return_tensors="pt").to(device)

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=1024,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
        return_dict_in_generate=True,
        output_scores=True,
        output_hidden_states=True,
    )

In [ ]:
# output.hidden_states: tuple[T] of tuple[num_layers+1] of [batch, seq_len, D]
# Step 0 has seq_len = full prompt length; later steps have seq_len = 1 (KV cache).
hidden_per_token = torch.stack([
    step[-1][0, -1, :].float()
    for step in output.hidden_states
])  # [T, D]

drift = representation_drift(hidden_per_token)
l2_drift     = drift["l2"].cpu()
cosine_drift = drift["cosine"].cpu()

generated_ids      = output.sequences[0][inputs["input_ids"].shape[1]:]
scores             = torch.stack(output.scores).squeeze(1)
token_log_probs_dr = F.log_softmax(scores, dim=-1).gather(
    1, generated_ids.unsqueeze(1)
).squeeze(1).float()

decoded_tokens_dr = [tokenizer.decode([tid.item()]) for tid in generated_ids]
slopes_dr = rolling_slope(token_log_probs_dr, window=20).cpu()

T = len(generated_ids)
print(f"Generated {T} tokens | hidden dim: {hidden_per_token.shape[1]}")
print(f"L2 drift     — min {l2_drift.min():.3f}  max {l2_drift.max():.3f}")
print(f"Cosine drift — min {cosine_drift.min():.5f}  max {cosine_drift.max():.5f}")

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)

axes[0].plot(token_log_probs_dr.cpu().numpy(), linewidth=0.7, color="tab:blue", alpha=0.85)
axes[0].set_ylabel("Log prob")
axes[0].set_title("Per-token log probability")

axes[1].plot(slopes_dr.numpy(), linewidth=0.7, color="tab:orange")
axes[1].axhline(0, color="k", linestyle="--", linewidth=0.5)
axes[1].set_ylabel("Slope")
axes[1].set_title("Rolling log prob slope (window=20)")

drift_x = range(1, T)
axes[2].plot(drift_x, l2_drift.numpy(), linewidth=0.7, color="tab:green", alpha=0.85)
axes[2].set_ylabel("L2 drift")
axes[2].set_title("Hidden-state L2 drift  \u2016hₜ − hₜ₋₁\u2016₂")

axes[3].plot(drift_x, cosine_drift.numpy(), linewidth=0.7, color="tab:purple", alpha=0.85)
axes[3].set_xlabel("Token index")
axes[3].set_ylabel("Cosine drift")
axes[3].set_title("Hidden-state cosine drift  1 − cos(hₜ, hₜ₋₁)")

plt.tight_layout()
plt.show()

In [ ]:
from IPython.display import HTML, display
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np

# drift[i] = change from token i → token i+1; assign to the arriving token (i+1).
# Token 0 has no predecessor, so it gets NaN → gray.
T = len(decoded_tokens_dr)

def pad_drift(drift_tensor):
    """Prepend NaN so index i holds the drift *into* token i."""
    arr = np.empty(T)
    arr[0] = np.nan
    arr[1:] = drift_tensor.numpy()
    return arr

l2_aligned     = pad_drift(l2_drift)
cosine_aligned = pad_drift(cosine_drift)

def make_html_block(title, metric_arr, cmap_name, unit=""):
    cmap = plt.get_cmap(cmap_name)
    valid = metric_arr[~np.isnan(metric_arr)]
    vmin, vmax = float(valid.min()), float(valid.max())
    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)

    parts = []
    for tok, val in zip(decoded_tokens_dr, metric_arr):
        display_tok = (
            tok.replace("&", "&amp;")
               .replace("<", "&lt;")
               .replace(">", "&gt;")
               .replace("\n", "↵<br>")
        )
        if np.isnan(val):
            bg, fg = "#d8d8d8", "#000"
            tooltip = "n/a"
        else:
            rgba = cmap(norm(float(val)))
            bg = mcolors.to_hex(rgba)
            # pick black or white text depending on luminance
            r, g, b = rgba[:3]
            lum = 0.2126 * r + 0.7152 * g + 0.0722 * b
            fg = "#000" if lum > 0.45 else "#fff"
            tooltip = f"{val:.5f}{unit}"

        parts.append(
            f'<span title="{title}={tooltip}" style="background:{bg};color:{fg};'
            f'padding:1px 3px;border-radius:2px;font-family:monospace;font-size:13px;">'
            f'{display_tok}</span>'
        )

    legend = (
        f'<p style="font-family:sans-serif;font-size:12px;margin:6px 0 3px 0;">'
        f'<b>{title}</b> &mdash; '
        f'<span style="background:{mcolors.to_hex(cmap(0.0))};padding:1px 6px;border-radius:2px;">low</span> '
        f'→ '
        f'<span style="background:{mcolors.to_hex(cmap(1.0))};color:#fff;padding:1px 6px;border-radius:2px;">high</span>'
        f' &nbsp; range [{vmin:.4f}, {vmax:.4f}]{unit}'
        f'</p>'
    )
    body = (
        '<div style="line-height:2.6;word-wrap:break-word;white-space:pre-wrap;'
        'border:1px solid #ccc;padding:10px;border-radius:4px;'
        'max-height:500px;overflow-y:auto;">'
        + "".join(parts)
        + "</div>"
    )
    return legend + body

html_l2     = make_html_block("L2 drift ‖hₜ−hₜ₋₁‖₂",          l2_aligned,     "YlOrRd")
html_cosine = make_html_block("Cosine drift 1−cos(hₜ,hₜ₋₁)",   cosine_aligned,  "YlGnBu")

display(HTML(html_l2))
display(HTML(html_cosine))

## With HotpotQA

In [ ]:
from datasets import load_dataset

ds = load_dataset("hotpotqa/hotpot_qa", "fullwiki")

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
from IPython.display import HTML, display

entry    = ds["train"][0]
question = entry["question"]
answer   = entry["answer"]

ctx_parts = []
for title, sentences in zip(entry["context"]["title"], entry["context"]["sentences"]):
    ctx_parts.append(f"{title}: " + " ".join(sentences))
context_str = "\n".join(ctx_parts)[:3000]

prompt = (
    "Answer the following multi-hop question. Use the provided context.\n\n"
    f"Context:\n{context_str}\n\n"
    f"Question: {question}\n\n"
    "Think through the reasoning steps, then give a concise final answer."
)

print(f"Question: {question}")
print(f"Answer:   {answer}")

messages = [{"role": "user", "content": prompt}]
text = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True,
)
inputs = tokenizer(text, return_tensors="pt").to(device)

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=1024,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
        return_dict_in_generate=True,
        output_scores=True,
        output_hidden_states=True,
    )

In [ ]:
# ── metrics ───────────────────────────────────────────────────────────────────
generated_ids = output.sequences[0][inputs["input_ids"].shape[1]:]
scores        = torch.stack(output.scores).squeeze(1)

token_log_probs_hq = F.log_softmax(scores, dim=-1).gather(
    1, generated_ids.unsqueeze(1)
).squeeze(1).float()

hidden_per_token_hq = torch.stack([
    step[-1][0, -1, :].float() for step in output.hidden_states
])  # [T, D]

drift_hq  = representation_drift(hidden_per_token_hq)
l2_hq     = drift_hq["l2"].cpu()
cosine_hq = drift_hq["cosine"].cpu()

decoded_tokens_hq = [tokenizer.decode([tid.item()]) for tid in generated_ids]
slopes_hq = rolling_slope(token_log_probs_hq, window=20).cpu()
T_hq = len(generated_ids)

print(f"Generated {T_hq} tokens | hidden dim: {hidden_per_token_hq.shape[1]}")
print(f"L2 drift     — min {l2_hq.min():.3f}  max {l2_hq.max():.3f}")
print(f"Cosine drift — min {cosine_hq.min():.5f}  max {cosine_hq.max():.5f}")

# ── HTML helper ──────────────────────────────────────────────────────────────
def tokens_to_html(decoded_tokens, metric_arr, cmap, norm, title):
    parts = []
    for tok, val in zip(decoded_tokens, metric_arr):
        dt = (tok.replace("&", "&amp;").replace("<", "&lt;")
                 .replace(">", "&gt;").replace("\n", "↵<br>"))
        if np.isnan(val):
            bg, fg, tip = "#d8d8d8", "#000", "n/a"
        else:
            rgba = cmap(norm(float(val)))
            lum  = 0.2126*rgba[0] + 0.7152*rgba[1] + 0.0722*rgba[2]
            bg, fg, tip = mcolors.to_hex(rgba), ("#000" if lum > 0.45 else "#fff"), f"{val:.5f}"
        parts.append(
            f'<span title="{title}={tip}" style="background:{bg};color:{fg};padding:1px 3px;border-radius:2px;font-family:monospace;font-size:13px;">{dt}</span>'
        )
    valid = metric_arr[~np.isnan(metric_arr)]
    lo, hi = float(valid.min()), float(valid.max())
    legend = (
        f'<p style="font-family:sans-serif;font-size:12px;margin:8px 0 3px 0;"><b>{title}</b>'
        f' &mdash; <span style="background:{mcolors.to_hex(cmap(0.0))};padding:1px 6px;border-radius:2px;">low</span>'
        f' → <span style="background:{mcolors.to_hex(cmap(1.0))};color:#fff;padding:1px 6px;border-radius:2px;">high</span>'
        f' &nbsp; range [{lo:.4f}, {hi:.4f}]</p>'
    )
    body = (
        '<div style="line-height:2.6;word-wrap:break-word;white-space:pre-wrap;'
        'border:1px solid #ccc;padding:10px;border-radius:4px;max-height:500px;overflow-y:auto;">'
        + "".join(parts) + "</div>"
    )
    return legend + body

WINDOW = 20
slope_aligned_hq = np.full(T_hq, np.nan)
for i, s in enumerate(slopes_hq.numpy()):
    center = i + WINDOW // 2
    if center < T_hq:
        slope_aligned_hq[center] = s
valid_s = slope_aligned_hq[~np.isnan(slope_aligned_hq)]
vmax_s  = float(np.abs(valid_s).max()) if len(valid_s) else 1.0

display(HTML(tokens_to_html(
    decoded_tokens_hq, slope_aligned_hq,
    plt.get_cmap("RdBu_r"),
    mcolors.TwoSlopeNorm(vmin=-vmax_s, vcenter=0.0, vmax=vmax_s),
    "Log-prob slope",
)))

l2_arr = np.empty(T_hq); l2_arr[0] = np.nan; l2_arr[1:] = l2_hq.numpy()
display(HTML(tokens_to_html(
    decoded_tokens_hq, l2_arr,
    plt.get_cmap("YlOrRd"),
    mcolors.Normalize(vmin=float(l2_hq.min()), vmax=float(l2_hq.max())),
    "L2 drift \u2016hₜ−hₜ₋₁\u2016₂",
)))

cos_arr = np.empty(T_hq); cos_arr[0] = np.nan; cos_arr[1:] = cosine_hq.numpy()
display(HTML(tokens_to_html(
    decoded_tokens_hq, cos_arr,
    plt.get_cmap("YlGnBu"),
    mcolors.Normalize(vmin=float(cosine_hq.min()), vmax=float(cosine_hq.max())),
    "Cosine drift 1−cos(hₜ,hₜ₋₁)",
)))

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)
fig.suptitle(f'HotpotQA — "{question[:80]}…"', fontsize=11)

drift_x_hq = range(1, T_hq)

axes[0].plot(token_log_probs_hq.cpu().numpy(), linewidth=0.7, color="tab:blue", alpha=0.85)
axes[0].set_ylabel("Log prob"); axes[0].set_title("Per-token log probability")

axes[1].plot(slopes_hq.numpy(), linewidth=0.7, color="tab:orange")
axes[1].axhline(0, color="k", linestyle="--", linewidth=0.5)
axes[1].set_ylabel("Slope"); axes[1].set_title("Rolling log prob slope (window=20)")

axes[2].plot(drift_x_hq, l2_hq.numpy(), linewidth=0.7, color="tab:green", alpha=0.85)
axes[2].set_ylabel("L2 drift"); axes[2].set_title("Hidden-state L2 drift  \u2016hₜ − hₜ₋₁\u2016₂")

axes[3].plot(drift_x_hq, cosine_hq.numpy(), linewidth=0.7, color="tab:purple", alpha=0.85)
axes[3].set_xlabel("Token index")
axes[3].set_ylabel("Cosine drift"); axes[3].set_title("Hidden-state cosine drift  1 − cos(hₜ, hₜ₋₁)")

plt.tight_layout()
plt.show()

In [ ]:
# ── shared HTML helpers ───────────────────────────────────────────────────────
def tokens_to_html(decoded_tokens, metric_arr, cmap, norm, title):
    parts = []
    for tok, val in zip(decoded_tokens, metric_arr):
        dt = (tok.replace("&", "&amp;").replace("<", "&lt;")
                 .replace(">", "&gt;").replace("\n", "↵<br>"))
        if np.isnan(val):
            bg, fg, tip = "#d8d8d8", "#000", "n/a"
        else:
            rgba = cmap(norm(float(val)))
            lum  = 0.2126*rgba[0] + 0.7152*rgba[1] + 0.0722*rgba[2]
            bg, fg, tip = mcolors.to_hex(rgba), ("#000" if lum > 0.45 else "#fff"), f"{val:.5f}"
        parts.append(
            f'<span title="{title}={tip}" style="background:{bg};color:{fg};'
            f'padding:1px 3px;border-radius:2px;font-family:monospace;font-size:13px;">{dt}</span>'
        )
    valid = metric_arr[~np.isnan(metric_arr)]
    lo, hi = float(valid.min()), float(valid.max())
    legend = (
        f'<p style="font-family:sans-serif;font-size:12px;margin:8px 0 3px 0;"><b>{title}</b>'
        f' &mdash; <span style="background:{mcolors.to_hex(cmap(0.0))};padding:1px 6px;border-radius:2px;">low</span>'
        f' → <span style="background:{mcolors.to_hex(cmap(1.0))};color:#fff;padding:1px 6px;border-radius:2px;">high</span>'
        f' &nbsp; range [{lo:.4f}, {hi:.4f}]</p>'
    )
    body = (
        '<div style="line-height:2.6;word-wrap:break-word;white-space:pre-wrap;'
        'border:1px solid #ccc;padding:10px;border-radius:4px;max-height:500px;overflow-y:auto;">'
        + "".join(parts) + "</div>"
    )
    return legend + body

# log-prob slope (diverging: blue=falling, red=rising)
WINDOW = 20
slope_aligned_hq = np.full(T_hq, np.nan)
for i, s in enumerate(slopes_hq.numpy()):
    center = i + WINDOW // 2
    if center < T_hq:
        slope_aligned_hq[center] = s
valid_s = slope_aligned_hq[~np.isnan(slope_aligned_hq)]
vmax_s  = float(np.abs(valid_s).max()) if len(valid_s) else 1.0

display(HTML(tokens_to_html(
    decoded_tokens_hq, slope_aligned_hq,
    plt.get_cmap("RdBu_r"),
    mcolors.TwoSlopeNorm(vmin=-vmax_s, vcenter=0.0, vmax=vmax_s),
    "Log-prob slope",
)))

# L2 drift (sequential: yellow=low, red=high)
l2_arr = np.empty(T_hq); l2_arr[0] = np.nan; l2_arr[1:] = l2_hq.numpy()
display(HTML(tokens_to_html(
    decoded_tokens_hq, l2_arr,
    plt.get_cmap("YlOrRd"),
    mcolors.Normalize(vmin=float(l2_hq.min()), vmax=float(l2_hq.max())),
    "L2 drift ‖hₜ−hₜ₋₁‖₂",
)))

# cosine drift (sequential: yellow=low, blue=high)
cos_arr = np.empty(T_hq); cos_arr[0] = np.nan; cos_arr[1:] = cosine_hq.numpy()
display(HTML(tokens_to_html(
    decoded_tokens_hq, cos_arr,
    plt.get_cmap("YlGnBu"),
    mcolors.Normalize(vmin=float(cosine_hq.min()), vmax=float(cosine_hq.max())),
    "Cosine drift 1−cos(hₜ,hₜ₋₁)",
)))

In [ ]:
import torch.nn.functional as F
import matplotlib.pyplot as plt

ALPHA = 5.0
WINDOW = 20
MAX_NEW_TOKENS = 1024

x_win = torch.arange(WINDOW, dtype=torch.float32, device=device) - (WINDOW - 1) / 2.0
denom = (x_win ** 2).sum()

input_ids_sg = inputs["input_ids"].clone()
past_kv = None
log_prob_history = []
gen_ids = []

for step in range(MAX_NEW_TOKENS):
    with torch.no_grad():
        if past_kv is None:
            out = model(input_ids=input_ids_sg, use_cache=True)
        else:
            out = model(
                input_ids=input_ids_sg[:, -1:],
                past_key_values=past_kv,
                use_cache=True,
            )
        past_kv = out.past_key_values
        logits = out.logits[0, -1]

    log_probs = F.log_softmax(logits, dim=-1)

    if len(log_prob_history) >= WINDOW - 1:
        recent = torch.tensor(
            log_prob_history[-(WINDOW - 1):], dtype=torch.float32, device=device
        )
        hist_dot = (x_win[:-1] * recent).sum()
        bonus = ALPHA * (hist_dot + x_win[-1] * log_probs) / denom
    else:
        bonus = torch.zeros_like(logits)

    probs = F.softmax(logits + bonus, dim=-1)
    next_id = torch.multinomial(probs, 1)

    log_prob_history.append(log_probs[next_id].item())
    gen_ids.append(next_id.item())
    input_ids_sg = next_id.unsqueeze(0)

    if next_id.item() == tokenizer.eos_token_id:
        break

response_sg = tokenizer.decode(torch.tensor(gen_ids), skip_special_tokens=True).strip()

print(f"Question : {question}")
print(f"Answer   : {answer}")
print(f"Generated: {len(gen_ids)} tokens")
print(f"Response : {response_sg}")

lp_tensor_sg = torch.tensor(log_prob_history, dtype=torch.float32)
slopes_sg    = rolling_slope(lp_tensor_sg, window=WINDOW)

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
fig.suptitle(f"Slope-guided generation (α={ALPHA}, window={WINDOW})  —  HotpotQA", fontsize=11)

axes[0].plot(log_prob_history, linewidth=0.7, color="tab:blue", alpha=0.85)
axes[0].set_ylabel("Log prob")
axes[0].set_title("Per-token log probability (slope-guided sampling)")

axes[1].plot(slopes_sg.numpy(), linewidth=0.7, color="tab:orange")
axes[1].axhline(0, color="k", linestyle="--", linewidth=0.5)
axes[1].set_xlabel("Token index")
axes[1].set_ylabel("Slope")
axes[1].set_title(f"Rolling log prob slope (window={WINDOW})")

plt.tight_layout()
plt.show()

In [ ]:
print(response_sg)